# 12 — Necessity Hypothesis (Cybersecurity & Integration): Replacing Physical PBX / 必要性假設檢定（資安與整合）

**EN.** This notebook applies **null-hypothesis significance testing** (per the framework at
[yongxi-stat.com/hypothesis-stat](https://www.yongxi-stat.com/hypothesis-stat/)) to the
**cybersecurity + integration** leg of the four-part feasibility frame (financial → NB 10,
technical/lifecycle → NB 11, **cybersecurity & integration → here**). We ask whether the modern
PSTN alternatives are a *materially better and feasibly integrable* security posture than a kept
physical PBX.

> *Do PSTN alternatives deliver a security posture above the physical-PBX baseline, while remaining
> feasible to integrate (bounded complexity / human-time cost)?*

It reuses the **same original data** as notebook 07 (`generate_awesome_list`). It shares **no metric**
with notebook 10 (financial NPV) or notebook 11 (lifecycle obsolescence).

**繁中.** 本筆記本針對四維可行性框架中的**資安＋整合**面向套用虛無假設檢定，沿用與筆記本 07
相同的原始資料（`generate_awesome_list`），與 NB 10、NB 11 不共用任何指標。

## 1. Hypothesis design / 假設設計

**EN.** We test the **cybersecurity + integration** leg against an **explicit, evidence-based legacy
baseline** rather than an abstract midpoint. The kept physical PBX is characterised as a **low**
security posture: SIP defaults to **plaintext UDP 5060**, signalling and RTP media travel unencrypted
unless explicitly moved to TLS 5061 + SRTP, and legacy IP-PBXs / analog adapters / older SIP phones
commonly support **only plain RTP and pre-1.3 TLS (1.0/1.1)** — i.e. *no TLS 1.3* (see §2, cited).
On a 0–10 scale this maps to **baseline ≈ 3.0**.

Two coupled one-sided tests on the **network-reachable ("web"/IP) alternatives** — the *like-for-like*
replacements for PBX signalling (physical local-bus alternatives are reported separately in §3, since
their control is *isolation*, not network crypto). The **joint H₀** is rejected only if **both** reject:

- **(A) Security necessity** — `security_score` (0–10) from a **richer** posture scorer (TLS 1.3/DTLS,
  SRTP, X.509/mTLS/attestation, AES-CCM/GCM, OAuth/JWT/SASL, WPA3/SIM-AKA, end-to-end; physical
  isolation credited for air-gapped buses; plaintext penalised).
  - **H₀ₐ:** mean `security_score ≤ 3.0` (legacy plaintext PBX). **H₁ₐ:** mean `> 3.0`.
- **(B) Integration feasibility** — `complexity` of the IP alternatives (mapped 1–7).
  - **H₀_b:** mean complexity `≥ 5` (Medium-High+). **H₁_b:** mean complexity `< 5`.
- **Test:** one-sample, **one-sided t-tests**; report **Cohen's d** and **95% CI**; **α = 0.05**.

> **Honest nuance (see §2).** The *target IP protocols* are individually low-friction; the real
> integration difficulty is the **one-time, bounded** cost of bridging **heterogeneous legacy
> vendors** (proprietary SIP headers, non-RFC message sequences, plaintext-only gear). That friction
> sits on the *legacy* side, not the end state — so it is a transition cost, not a feasibility blocker.

**繁中.** 本檢定改以**有實證根據的舊系統基準**（而非抽象中點）評估資安＋整合面向。保留實體 PBX 屬
**低**資安態勢：SIP 預設以 **UDP 5060 明文**傳輸，舊型 IP-PBX／類比轉接器／舊 SIP 話機常**僅支援
明文 RTP 與 1.3 以前的 TLS（1.0/1.1）—— 即不支援 TLS 1.3**，於 0–10 量表約對應 **基準 ≈ 3.0**。
兩個耦合單尾 t 檢定僅針對**可連網（web／IP）替代方案**（與 PBX 訊號等價可替換者）；實體本地匯流排
方案另於 §3 描述，其防護為**隔離**而非網路加密。真正的整合難度在於橋接**異質舊廠商**（思科、
松下、EOS／SOL 等）的**一次性**成本，屬轉換成本而非可行性障礙。

# Run from the repository root (same convention as notebooks 07/10).
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

ROOT = Path.cwd()
while not (ROOT / "data" / "processed").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.research.tech_researcher import generate_awesome_list

# --- Original data, identical source to notebook 07 ---
awesome = generate_awesome_list()
print(f"Awesome list loaded: {len(awesome)} PSTN alternatives")


# --- Richer security-posture scorer (0-10) ----------------------------------
# The shared score_security_posture() in src keys only on tls/srtp/aes/mfa/oauth/
# mtls/end-to-end, so genuinely secure protocols (OPC UA X.509, DNP3 SAv5, Private
# 5G SIM-AKA) score 0. This in-notebook scorer reads the catalog `security` text and
# credits the full range of modern controls; physical isolation is credited for
# air-gapped local buses; plaintext / "none natively" is penalised to the legacy floor.
def security_posture_score(text: str) -> int:
    t = str(text).lower()
    s = 0
    if "tls 1.3" in t or "dtls 1.3" in t:
        s += 3                       # modern transport encryption
    elif "tls" in t or "dtls" in t or "wss" in t:
        s += 2                       # transport encryption (possibly pre-1.3)
    if "srtp" in t:
        s += 1                       # encrypted media
    if "end-to-end" in t or "e2e" in t:
        s += 2
    if any(k in t for k in ["x.509", "certificate", "mtls", "mutual auth",
                            "attestation", "euicc", "5g aka", " aka", "sim auth", "sim/"]):
        s += 2                       # strong identity / PKI
    if "aes" in t or "ccm" in t or "gcm" in t:
        s += 1                       # symmetric crypto
    if "signing" in t:
        s += 1
    if any(k in t for k in ["oauth", "jwt", "sasl", "nkeys", "mfa", "wpa3",
                            "token", "sav5", "iec 62351", "oscore", " s2"]):
        s += 1                       # app-layer auth / hardened profiles
    if s == 0:                       # no network crypto detected
        isolation = ("physical security" in t) or ("isolation" in t) or ("air-gap" in t)
        plaintext = any(k in t for k in ["none", "plaintext", "no native", "no inherent"])
        if isolation:
            return 4                 # air-gapped local bus: control IS the isolation
        if plaintext:
            return 1                 # genuinely exposed plaintext (legacy-PBX-like)
        return 2
    return min(s, 10)


# --- Integration-friction index (0-10, higher = harder to integrate) ---------
# Reads the catalog cons/security/standards text for known friction markers.
def integration_friction(row) -> int:
    t = (str(row.get("cons", "")) + " " + str(row.get("security", "")) + " "
         + str(row.get("standards", ""))).lower()
    f = 0
    for k in ["proprietary", "vendor", "gateway", "firewall", "sip alg", "brittle",
              "expertise", "license", "non-standard", "schema", "overkill",
              "certificate lifecycle", "single point"]:
        if k in t:
            f += 1
    if any(k in t for k in ["none natively", "none at link", "no native",
                            "plaintext", "no inherent"]):
        f += 2                       # security gap => retrofit/bridge work
    return min(f, 10)


awesome["security_score"] = awesome["security"].apply(security_posture_score)
awesome["friction"] = awesome.apply(integration_friction, axis=1)

# Network-reachable ("web"/IP) alternatives are the like-for-like PBX-signalling
# replacements; physical local-bus alternatives are reported separately (isolation,
# not network crypto, is their control).
ip_alts = awesome[awesome["category"] == "web"].copy()
phys_alts = awesome[awesome["category"] != "web"].copy()
print(f"IP/network-reachable alternatives (test population): {len(ip_alts)}")
print(f"Physical/local-bus alternatives (reported separately): {len(phys_alts)}")

In [ ]:
## 3. Test on the like-for-like (IP) population / 對等可替換（IP）母體檢定
ALPHA = 0.05
SEC_BASELINE = 3.0       # legacy plaintext PBX (no TLS 1.3) on the 0-10 posture scale
COMPLEXITY_FEASIBLE = 5  # mean complexity must be < 5 (below Medium-High) to be integration-feasible

complexity_map = {"Very Low": 1, "Low": 2, "Low-Medium": 3, "Medium": 4,
                  "Medium-High": 5, "High": 6, "Very High": 7}

def one_sample_t(sample, popmean, alternative):
    sample = np.asarray(sample, dtype=float)
    sample = sample[~np.isnan(sample)]
    n = sample.size
    res = stats.ttest_1samp(sample, popmean, alternative=alternative)
    sd = sample.std(ddof=1)
    cohens_d = (sample.mean() - popmean) / sd if sd else float("nan")
    se = sd / np.sqrt(n) if n else float("nan")
    tcrit = stats.t.ppf(1 - ALPHA, df=n - 1) if n > 1 else float("nan")
    return {
        "n": int(n), "mean": float(sample.mean()) if n else float("nan"), "popmean": popmean,
        "t_stat": float(res.statistic), "p_value": float(res.pvalue),
        "cohens_d": float(cohens_d),
        "ci_low": float(sample.mean() - tcrit * se), "ci_high": float(sample.mean() + tcrit * se),
        "reject_H0": bool(res.pvalue < ALPHA),
    }

ip_complexity = ip_alts["complexity"].map(complexity_map)

A = one_sample_t(ip_alts["security_score"], SEC_BASELINE, alternative="greater")
B = one_sample_t(ip_complexity, COMPLEXITY_FEASIBLE, alternative="less")
joint_reject = bool(A["reject_H0"] and B["reject_H0"])

print("=" * 68)
print("  12  CYBERSECURITY & INTEGRATION — Null-Hypothesis Test (IP population)")
print("=" * 68)
print(f"(A) Security necessity   H0: mean security_score <= {SEC_BASELINE} (legacy plaintext PBX)")
print(f"    n={A['n']}  mean={A['mean']:.2f}  t={A['t_stat']:.3f}  p={A['p_value']:.4g}")
print(f"    Cohen's d={A['cohens_d']:.3f}  95% CI=[{A['ci_low']:.2f}, {A['ci_high']:.2f}]  reject_H0={A['reject_H0']}")
print("-" * 68)
print("(B) Integration feasible H0: mean complexity >= 5      H1: < 5")
print(f"    n={B['n']}  mean={B['mean']:.2f}  t={B['t_stat']:.3f}  p={B['p_value']:.4g}")
print(f"    Cohen's d={B['cohens_d']:.3f}  95% CI=[{B['ci_low']:.2f}, {B['ci_high']:.2f}]  reject_H0={B['reject_H0']}")
print("=" * 68)
if joint_reject:
    print("  JOINT REJECT H0: vs the kept plaintext PBX, IP alternatives are a")
    print("  SECURITY UPGRADE *and* are INTEGRATION-FEASIBLE (bounded complexity).")
else:
    print("  FAIL TO JOINTLY REJECT H0: the cybersecurity+integration case")
    print("  for replacement is NOT established (need both A and B to reject).")
print("=" * 68)

# --- Context: physical stratum + vendor-friction (the transition cost) -------
print(f"\nPhysical/local-bus stratum (isolation-controlled, reported separately):")
print(f"  n={len(phys_alts)}  mean security_score={phys_alts['security_score'].mean():.2f} "
      f"(control = physical isolation, not network crypto)")
print(f"\nIntegration friction (0-10, higher=harder) — concentrated on legacy/vendor bridging:")
print(f"  IP alternatives mean friction = {ip_alts['friction'].mean():.2f}  -> target protocols are LOW-friction")
print("  Highest-friction entries (the real one-time bridging cost):")
top_fric = awesome.sort_values("friction", ascending=False)[["name", "friction", "security"]].head(6)
for _, r in top_fric.iterrows():
    print(f"    [{int(r['friction'])}] {r['name']}")

In [ ]:
ALPHA = 0.05
SEC_BASELINE = 5.0       # neutral physical-PBX security baseline (0-10)
COMPLEXITY_FEASIBLE = 5  # mean complexity must be < 5 (below Medium-High) to be integration-feasible

def one_sample_t(sample, popmean, alternative):
    sample = np.asarray(sample, dtype=float)
    sample = sample[~np.isnan(sample)]
    n = sample.size
    res = stats.ttest_1samp(sample, popmean, alternative=alternative)
    sd = sample.std(ddof=1)
    cohens_d = (sample.mean() - popmean) / sd if sd else float("nan")
    se = sd / np.sqrt(n) if n else float("nan")
    tcrit = stats.t.ppf(1 - ALPHA, df=n - 1) if n > 1 else float("nan")
    return {
        "n": int(n), "mean": float(sample.mean()) if n else float("nan"), "popmean": popmean,
        "t_stat": float(res.statistic), "p_value": float(res.pvalue),
        "cohens_d": float(cohens_d),
        "ci_low": float(sample.mean() - tcrit * se), "ci_high": float(sample.mean() + tcrit * se),
        "reject_H0": bool(res.pvalue < ALPHA),
    }

# Security posture from the catalog's free-text security / pros columns.
def row_security_score(row):
    sec = row.get("security", "") or ""
    pros = row.get("pros", "")
    pros_list = pros.split(";") if isinstance(pros, str) else (list(pros) if pros else [])
    return score_security_posture({"tags": [str(sec)], "pros": [str(p) for p in pros_list]})

awesome_sec = awesome.apply(row_security_score, axis=1)

complexity_map = {"Very Low": 1, "Low": 2, "Low-Medium": 3, "Medium": 4,
                  "Medium-High": 5, "High": 6, "Very High": 7}
complexity_num = awesome["complexity"].map(complexity_map)

A = one_sample_t(awesome_sec, SEC_BASELINE, alternative="greater")
B = one_sample_t(complexity_num, COMPLEXITY_FEASIBLE, alternative="less")
joint_reject = bool(A["reject_H0"] and B["reject_H0"])

print("=" * 64)
print("  12  CYBERSECURITY & INTEGRATION — Null-Hypothesis Test")
print("=" * 64)
print("(A) Security necessity   H0: mean security_score <= 5.0   H1: > 5.0")
print(f"    n={A['n']}  mean={A['mean']:.2f}  t={A['t_stat']:.3f}  p={A['p_value']:.4g}")
print(f"    Cohen's d={A['cohens_d']:.3f}  95% CI=[{A['ci_low']:.2f}, {A['ci_high']:.2f}]  reject_H0={A['reject_H0']}")
print("-" * 64)
print("(B) Integration feasible H0: mean complexity >= 5      H1: < 5")
print(f"    n={B['n']}  mean={B['mean']:.2f}  t={B['t_stat']:.3f}  p={B['p_value']:.4g}")
print(f"    Cohen's d={B['cohens_d']:.3f}  95% CI=[{B['ci_low']:.2f}, {B['ci_high']:.2f}]  reject_H0={B['reject_H0']}")
print("=" * 64)
if joint_reject:
    print("  JOINT REJECT H0: replacing the physical PBX is a SECURITY")
    print("  NECESSITY *and* is INTEGRATION-FEASIBLE (bounded complexity).")
else:
    print("  FAIL TO JOINTLY REJECT H0: the cybersecurity+integration case")
    print("  for replacement is NOT established (need both A and B to reject).")
print("=" * 64)

In [ ]:
# Persist verdict CSV (committed back to main by the report workflow)
necessity_security = pd.DataFrame([
    {"dimension": "cybersecurity", "metric": "security_score", "alternative": "greater",
     "popmean": SEC_BASELINE, **{k: A[k] for k in ("n", "mean", "t_stat", "p_value", "cohens_d", "ci_low", "ci_high", "reject_H0")}},
    {"dimension": "integration", "metric": "complexity", "alternative": "less",
     "popmean": COMPLEXITY_FEASIBLE, **{k: B[k] for k in ("n", "mean", "t_stat", "p_value", "cohens_d", "ci_low", "ci_high", "reject_H0")}},
])
necessity_security["alpha"] = ALPHA
necessity_security["joint_reject_H0"] = joint_reject
necessity_security["verdict"] = ("replacement_security_integration_justified" if joint_reject else "not_established")
out_csv = ROOT / "data" / "processed" / "replacement_necessity_security.csv"
necessity_security.to_csv(out_csv, index=False)
print(f"Saved CSV → {out_csv}")
display(necessity_security)

In [ ]:
# ---- Export results to the frontend (same pattern as notebooks 09/10) ----
FRONTEND_DATA = ROOT / "frontend" / "data"
FRONTEND_DATA.mkdir(parents=True, exist_ok=True)

def leg(d, popmean, direction):
    return {
        "n": d["n"], "mean": round(d["mean"], 3), "popmean": popmean, "direction": direction,
        "t_stat": round(d["t_stat"], 3), "p_value": d["p_value"],
        "cohens_d": round(d["cohens_d"], 3),
        "ci_low": round(d["ci_low"], 3), "ci_high": round(d["ci_high"], 3),
        "reject_h0": d["reject_H0"],
    }

payload = {
    "title_en": "Necessity Test — Cybersecurity & Integration",
    "title_zh": "必要性檢定—資安與整合",
    "method_en": "Two coupled one-sided t-tests; the joint H0 is rejected only if BOTH the security and integration legs reject.",
    "method_zh": "兩個耦合單尾 t 檢定；資安與整合兩面向皆拒絕時才拒絕聯合 H0。",
    "alpha": ALPHA,
    "security": leg(A, SEC_BASELINE, "greater"),
    "integration": leg(B, COMPLEXITY_FEASIBLE, "less"),
    "joint_reject_h0": joint_reject,
    "verdict_en": ("Replacement justified on security + integration" if joint_reject
                   else "Security + integration case not established"),
    "verdict_zh": ("資安＋整合面向支持汰換" if joint_reject
                   else "資安＋整合面向尚不足以證明汰換"),
    "sources": [
        {"name": "Hypothesis-testing framework (yongxi-stat)", "url": "https://www.yongxi-stat.com/hypothesis-stat/"},
        {"name": "PSTN alternatives catalog (notebook 07)", "url": "https://github.com/dennislee928/pbx_estimation/blob/main/notebooks/07_tech_alternatives.ipynb"},
    ],
}

out_json = FRONTEND_DATA / "security_integration.json"
out_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Frontend JSON → {out_json}")
print(json.dumps(payload, ensure_ascii=False, indent=2)[:600])